# MLOps Training 2026/2027 — Task 2
## Notebook 3: Train, Validation, and Test Split
In this notebook, I will split the labeled dataset into training, validation, and test sets before doing detailed EDA.

I will first check the order date range and target distribution to decide whether a random or time-based split is more suitable. The test set will be kept separate and will not be used during EDA or model development.

## 1. Load the Labeled Dataset
I will load the labeled table created in Notebook 2 and check its size before deciding how to split the data.

In [9]:
import pandas as pd

labeled_table = pd.read_csv(
    "artifacts/labeled_table.csv",
    parse_dates=["order_purchase_timestamp"]
)

print("Rows:", labeled_table.shape[0])
print("Columns:", labeled_table.shape[1])

labeled_table.head()

Rows: 96476
Columns: 30


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_payment_value,max_installments,payment_type_count,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude,distance_km,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,38.71,1.0,2.0,9350.0,maua,SP,-23.680729,-46.444238,18.576110,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,141.46,1.0,1.0,31570.0,belo horizonte,SP,-19.807681,-43.980427,851.495069,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,179.12,3.0,1.0,14840.0,guariba,SP,-21.363502,-48.229601,514.410666,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,72.20,1.0,1.0,31842.0,belo horizonte,MG,-19.837682,-43.924053,1822.226336,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,28.62,1.0,1.0,8752.0,mogi das cruzes,SP,-23.543395,-46.262086,29.676625,0


## 2. Choose the Split Method
Before splitting the data, I will check the purchase date range and the late-delivery rate over time. This will help decide whether a random or time-based split is more appropriate.

In [10]:
print("Earliest order:", labeled_table["order_purchase_timestamp"].min())
print("Latest order:", labeled_table["order_purchase_timestamp"].max())
print(
    "Time span:",
    labeled_table["order_purchase_timestamp"].max()
    - labeled_table["order_purchase_timestamp"].min()
)

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37
Time span: 713 days 02:43:59


### 2.1 Delivery Pattern Over Time
I will check the number of orders and the late-delivery rate by month to see whether the pattern changes over time.

In [11]:
monthly_summary = (
    labeled_table
    .assign(
        order_month=labeled_table["order_purchase_timestamp"].dt.to_period("M")
    )
    .groupby("order_month")
    .agg(
        orders=("order_id", "count"),
        late_rate=("late_delivery", "mean")
    )
    .reset_index()
)

monthly_summary["late_rate"] = (
    monthly_summary["late_rate"] * 100
).round(2)

monthly_summary

,order_month,orders,late_rate
0,2016-09,1,100.00
1,2016-10,270,0.74
2,2016-12,1,0.00
3,2017-01,750,2.93
4,2017-02,1653,2.96
5,2017-03,2546,4.56
6,2017-04,2303,6.56
7,2017-05,3545,2.99
8,2017-06,3135,3.03
9,2017-07,3872,2.79


### 2.2 Split Decision
The late-delivery rate changes over time, so I will use a time-based split instead of a random split.

This also matches the real prediction setting: the model should learn from past orders and then be evaluated on newer orders that it has not seen before.

The data will be sorted by `order_purchase_timestamp` before creating the training, validation, and test sets.

In [12]:
labeled_table = (
    labeled_table
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

labeled_table[
    ["order_purchase_timestamp", "late_delivery"]
].head()

,order_purchase_timestamp,late_delivery
0,2016-09-15 12:16:38,1
1,2016-10-03 09:44:50,0
2,2016-10-03 16:56:50,0
3,2016-10-03 21:01:41,0
4,2016-10-03 21:13:36,0


## 3. Create the Train, Validation, and Test Sets
I will use a 70/15/15 time-based split:

- 70% of the earliest orders for training
- 15% of the following orders for validation
- 15% of the newest orders for testing

The data will not be shuffled so that the chronological order is preserved.

In [13]:
n = len(labeled_table)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = labeled_table.iloc[:train_end].copy()
validation = labeled_table.iloc[train_end:val_end].copy()
test = labeled_table.iloc[val_end:].copy()

print("Total rows:", n)
print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))

Total rows: 96476
Training rows: 67533
Validation rows: 14471
Test rows: 14472


### 3.1 Check the Splits
I will check the date range and late-delivery rate in each split to make sure the chronological order is preserved and each split contains both target classes.

In [14]:
split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "rows": [
        len(train),
        len(validation),
        len(test)
    ],
    "start_date": [
        train["order_purchase_timestamp"].min(),
        validation["order_purchase_timestamp"].min(),
        test["order_purchase_timestamp"].min()
    ],
    "end_date": [
        train["order_purchase_timestamp"].max(),
        validation["order_purchase_timestamp"].max(),
        test["order_purchase_timestamp"].max()
    ],
    "late_orders": [
        train["late_delivery"].sum(),
        validation["late_delivery"].sum(),
        test["late_delivery"].sum()
    ],
    "late_rate": [
        train["late_delivery"].mean() * 100,
        validation["late_delivery"].mean() * 100,
        test["late_delivery"].mean() * 100
    ]
})

split_summary["late_rate"] = split_summary["late_rate"].round(2)

split_summary

,split,rows,start_date,end_date,late_orders,late_rate
0,Train,67533,2016-09-15 12:16:38,2018-04-15 20:07:56,5291,7.83
1,Validation,14471,2018-04-15 20:10:23,2018-06-21 07:50:39,624,4.31
2,Test,14472,2018-06-21 08:29:29,2018-08-29 15:00:37,620,4.28


The chronological order is preserved across the three splits. The training set contains the earliest orders, followed by validation, while the test set contains the newest orders.

The late-delivery rate is higher in the training set (7.83%) than in validation (4.31%) and test (4.28%). This difference reflects the change in delivery patterns over time and is kept as part of the time-based split.

## 4. Save the Split Datasets
The train, validation, and test sets will be saved as separate artifacts for the next notebooks.

In [15]:
train.to_csv("artifacts/train.csv", index=False)
validation.to_csv("artifacts/validation.csv", index=False)
test.to_csv("artifacts/test.csv", index=False)

print("Train, validation, and test files saved successfully!")

Train, validation, and test files saved successfully!


## 5. Final Check
Before finishing this notebook, I will load the saved files and confirm their sizes, order IDs, and chronological order.

In [16]:
saved_train = pd.read_csv(
    "artifacts/train.csv",
    parse_dates=["order_purchase_timestamp"]
)

saved_validation = pd.read_csv(
    "artifacts/validation.csv",
    parse_dates=["order_purchase_timestamp"]
)

saved_test = pd.read_csv(
    "artifacts/test.csv",
    parse_dates=["order_purchase_timestamp"]
)

print("Train rows:", len(saved_train))
print("Validation rows:", len(saved_validation))
print("Test rows:", len(saved_test))

print("\nDuplicate order IDs:")
print("Train:", saved_train["order_id"].duplicated().sum())
print("Validation:", saved_validation["order_id"].duplicated().sum())
print("Test:", saved_test["order_id"].duplicated().sum())

print("\nChronological boundaries:")
print("Train ends:", saved_train["order_purchase_timestamp"].max())
print("Validation starts:", saved_validation["order_purchase_timestamp"].min())
print("Validation ends:", saved_validation["order_purchase_timestamp"].max())
print("Test starts:", saved_test["order_purchase_timestamp"].min())

Train rows: 67533
Validation rows: 14471
Test rows: 14472

Duplicate order IDs:
Train: 0
Validation: 0
Test: 0

Chronological boundaries:
Train ends: 2018-04-15 20:07:56
Validation starts: 2018-04-15 20:10:23
Validation ends: 2018-06-21 07:50:39
Test starts: 2018-06-21 08:29:29


## 6. Conclusion
The labeled dataset was split chronologically into 70% training, 15% validation, and 15% test data.

A time-based split was used because the delivery pattern changes over time and it better represents training on past orders and predicting newer orders. The test set will remain separate from the detailed EDA and model development.

The three datasets were saved as `train.csv`, `validation.csv`, and `test.csv` for the next steps.